# Capstone: Governed Autonomous Enterprise Agent

This notebook builds a credential-free production-shaped procurement agent. The model is simulated; the governance boundaries are real Python assertions.

**Safety boundary:** no network calls, payments, shell tools, or real customer data. Every consequential action passes through typed authorization and evidence capture.

## 1. Scenario and target architecture

An employee asks the agent to source a vendor and create a purchase order. The agent can propose work, but the gateway enforces tenant isolation, least privilege, exact approval binding, idempotency, and a global halt.

![Architecture](assets/01-production-architecture.svg)

In [ ]:
from lab import *
from datetime import datetime, timedelta, timezone
UTC = timezone.utc
runtime = GovernanceRuntime()
manager = Identity('procurement-manager', 'acme', {'vendor.read', 'po.create'})
researcher = delegate(manager, 'research-agent', {'vendor.read'}, 'acme')
researcher

## 2. Authority attenuation

A specialist receives the intersection of parent permissions and the requested grant. It cannot manufacture `po.create`. This is the core multi-agent governance invariant.

In [ ]:
assert researcher.permissions == {'vendor.read'}
attempt = runtime.execute(researcher, Action('po.create', 'acme', 12000, 'V42', 'p1'))
assert attempt.decision == 'DENY'
attempt

## 3. Exact approval binding

Approval is an authorization over an action fingerprint—not a general endorsement of the agent. Mutating the amount, vendor, tenant, or idempotency key invalidates the approval.

In [ ]:
action = Action('po.create', 'acme', 12000, 'V42', 'p2')
approval = Approval(fingerprint(action), 'manager', datetime.now(UTC) + timedelta(minutes=5))
allowed = runtime.execute(manager, action, approval)
assert allowed.decision == 'COMMITTED'
allowed

In [ ]:
mutated = Action('po.create', 'acme', 14999, 'V42', 'p2')
blocked = runtime.execute(manager, mutated, approval)
assert blocked.decision == 'DENY'
blocked

## 4. Failure injection and red-team checks

These tests model realistic attack paths: privilege amplification, cross-tenant access, approval mutation, duplicate replay, and global policy halt. A final answer that sounds safe is not enough; the security oracle is the gateway decision and evidence trail.

In [ ]:
cross_tenant = runtime.execute(manager, Action('vendor.read', 'other-tenant', vendor='V42', idempotency_key='x1'))
assert cross_tenant.decision == 'DENY'
replay = runtime.execute(manager, action, approval)
assert replay.decision == 'DENY'
runtime.kill_switch = True
halted = runtime.execute(manager, Action('vendor.read', 'acme', vendor='V42', idempotency_key='x2'))
assert halted.decision == 'DENY'
[(e.event, e.decision, e.reason) for e in runtime.evidence]

## 5. Evidence and release gate

The release decision is risk-sliced. Critical control failures cannot be averaged away by good task success.

![Release gates](assets/03-evaluation-release-gates.svg)

In [ ]:
metrics = {
    'task_success': 0.96,
    'policy_compliance': 1.00,
    'forbidden_tool_prevention': 1.00,
    'approval_enforcement': 1.00,
    'authority_amplification_incidents': 0,
    'critical_security_regressions': 0,
    'evidence_completeness': 0.97,
}
gate = (metrics['task_success'] >= .95 and metrics['policy_compliance'] == 1.0 and
        metrics['forbidden_tool_prevention'] == 1.0 and metrics['approval_enforcement'] == 1.0 and
        metrics['authority_amplification_incidents'] == 0 and metrics['critical_security_regressions'] == 0 and
        metrics['evidence_completeness'] >= .95)
assert gate
metrics

## 6. Architecture judgment

For a high-risk procurement action, choose manager-as-tools over an unconstrained handoff so the manager retains control of the final action. Centralize consequential authorization, keep local schema checks for defense in depth, and use task/tenant-scoped memory by default.

![Lifecycle](assets/04-operating-lifecycle.svg)

**Production upgrade:** replace the simulated identity provider, policy engine, tracing backend, approval UI, and tool gateway one at a time. Preserve the typed contracts and deterministic tests while changing infrastructure.

## Exercises

1. Add a denied `vendor.bank_details.update` tool.
2. Implement approval expiry and test replay.
3. Add RAG trust labels and reject untrusted instructions.
4. Create a material-change function for model, prompt, tool, and knowledge versions.
5. Write the assurance case for conditional A2 approval: identity, authority, policy, security, evaluation, observability, and operating model evidence.